<a href="https://colab.research.google.com/github/Khushi-P-2004/Machine-Learning-Project/blob/main/23f1002421_notebook_t12026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

comment_category_prediction_challenge_path = kagglehub.competition_download('comment-category-prediction-challenge')

print('Data source import complete.')


## 1. Problem Statement
####  Explore the dataset and build predictive models that can accurately determine the final category assigned to each comment.

## 2. Import libraries

In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import f1_score, confusion_matrix, classification_report

## 3. Load dataset

In [ ]:
train_df = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
sample_df = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
train_df.dtypes

#### The dataset contains 198000 rows and 15 features including text, numerical, categorical and boolean types.

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.isnull().mean()*100

#### race, religion, gender have ~73% missing values

In [ ]:
train_df['label'].value_counts()

In [ ]:
train_df['label'].value_counts(normalize=True)

In [ ]:
sns.countplot(x='label', data=train_df)
plt.title("class distribution")
plt.show()

#### The dataset is highly imbalanced with ~57% class 0 and class 3 being rare with ~2.7%

In [ ]:
train_df['comment'].str.len().describe()

#### This is the character length distribution of comments.

- #### The comment length distribution is positively skewed due to presence of long-tail comments

- #### high variation

- #### very long comments exist (outliers)

- #### some extremely short comments exist too

In [ ]:
sns.boxplot(x='label',y=train_df['comment'].str.len(),data=train_df)
plt.title('comment length vs label')
plt.show()

In [ ]:
train_df['comment'].str.split().str.len().describe()

#### This is the word count distribution

- #### word count distribution shows positive skewness with long-tail comments
- #### significant outliers in comment length and word counts

In [ ]:
from collections import Counter
Counter(" ".join(train_df['comment'].fillna("")).split()).most_common(20)

#### The most frequent words are stopwords. No strange tokens dominating. This justifies the use of stopword removal during TF-IDF vectorization to reduce noise and dimensionality.

In [ ]:
train_df['upvote'].describe()

In [ ]:
train_df['downvote'].describe()

In [ ]:
train_df.groupby('label')['upvote'].mean()

In [ ]:
train_df.groupby('label')['downvote'].mean()

In [ ]:
num_cols = ['if_1','if_2','upvote','downvote']
train_df[num_cols].hist(figsize=(15,8),bins=30,edgecolor='blue')
plt.show()

#### The numeric features show strong right skewness with most observations near zero
#### outliers exist representing highly engaged comments
#### Scaling need to be done to normalize feature distributions and reduce the impact of extreme values

## 5. Data Preprocessing

In [ ]:
X = train_df.drop(["label","post_id","created_date"], axis=1)
y = train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

test_df=test_df.drop(["post_id","created_date"],axis=1)

## 6. Feature Engineering

In [ ]:
for df in [X_train, X_test, test_df]:
    df['comment'] = df['comment'].fillna("")
    df['race']=df['race'].fillna("None")
    df['religion']=df['religion'].fillna("None")
    df['gender']=df['gender'].fillna("None")
    df['char_len']=df['comment'].str.len()
    df['word_count']=df['comment'].str.split().str.len()
    df['avg_word_len']=df['char_len']/(df['word_count']+1)
    df['caps_count']=df['comment'].str.count(r'[A-Z]')
    df['exclaim_count']=df['comment'].str.count('!')
    df['question_count']=df['comment'].str.count(r'\?')
    df['dots_count']=df['comment'].str.count(r'\.')
    df['specialchar_count']=df['comment'].str.count(r'[^A-Za-z0-9\s]')
    df['digit_count']=df['comment'].str.count(r'\d')
    df['total_votes']=df['upvote']+df['downvote']
    df['votes_diff']=df['upvote']-df['downvote']
    df['votes_ratio']=df['upvote']/(df['downvote']+1)
    df['short_comment']=(df['word_count']<20).astype(int)
    df['long_comment']=(df['word_count']>80).astype(int)
    df['is_high_engmt']=(df['total_votes']>50).astype(int)

#### Feature engineering was done to capture
- #### text structure such as length, word count
- #### emotional intensity with use of block letters and punctuations
- #### engagement (votes)

## 7. Feature Transformation

In [ ]:
numeric_f = ['upvote','downvote','if_1','if_2','char_len','word_count',
             'avg_word_len','caps_count','exclaim_count','question_count',
             'dots_count','specialchar_count','digit_count','total_votes',
             'votes_diff','votes_ratio']

word_tfidf = TfidfVectorizer(analyzer='word',
                             max_features=30000,
                             ngram_range=(1,2),
                             stop_words='english',
                             min_df=3,
                             sublinear_tf=True)

char_tfidf = TfidfVectorizer(analyzer='char',
                             max_features=15000,
                             ngram_range=(3,4),
                             min_df=5)

prep = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_f),
        ('bin', 'passthrough', ['emoticon_1','emoticon_2','emoticon_3',
                                'disability','short_comment','long_comment',
                                'is_high_engmt']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['race','religion','gender']),
        ('word_tfidf', word_tfidf, 'comment'),
        ('char_tfidf', char_tfidf, 'comment')
    ]
)

## 8. Model Training

### 8.1. Logistic Regression

In [ ]:
model_lr = LogisticRegression(
    C=50,
    solver='saga',
    penalty='l2',
    max_iter=4000,
    class_weight='balanced',
    n_jobs=-1
)

pipe_lr = Pipeline([
    ('preprocess', prep),
    ('model_lr', model_lr)
])

pipe_lr.fit(X_train,y_train)
lr_y_pred=pipe_lr.predict(X_test)
print("LR score:",f1_score(y_test, lr_y_pred, average='macro'))
print(confusion_matrix(y_test, lr_y_pred))

print("LR report")
lr_report=classification_report(y_test,lr_y_pred, output_dict=True)
pd.DataFrame(lr_report).transpose()

### 8.2. XGBoost

In [ ]:
model_xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

pipe_xgb = Pipeline([
    ('preprocess',prep),
    ('model',model_xgb)
])
pipe_xgb.fit(X_train,y_train)
xgb_pred = pipe_xgb.predict(X_test)
print("XGB score:",f1_score(y_test, xgb_pred, average='macro'))
print(confusion_matrix(y_test, xgb_pred))

print("XGB report")
xgb_report=classification_report(y_test,xgb_pred, output_dict=True)
pd.DataFrame(xgb_report).transpose()

### 8.3. SGDClassifier

In [ ]:
model_sgd = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=1e-4,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    class_weight='balanced',
    random_state=42
)
pipe_sgd = Pipeline([
    ('preprocess', prep),
    ('model', model_sgd)
])
pipe_sgd.fit(X_train,y_train)
sgd_y_pred=pipe_sgd.predict(X_test)
print(f1_score(y_test, sgd_y_pred, average='macro'))
print(confusion_matrix(y_test, sgd_y_pred))
print("SGD report")
sgd_report=classification_report(y_test,sgd_y_pred, output_dict=True)
pd.DataFrame(sgd_report).transpose()

## 9. Hyperparameter Tuning

In [ ]:
param_lr = {
    'model_lr__C':[20,50,80]
}
random_lr = RandomizedSearchCV(
    pipe_lr,
    param_distributions=param_lr,
    n_iter=2,
    scoring='f1_macro',
    cv=2,
    n_jobs=-1,
    verbose=True
)
random_lr.fit(X_train,y_train)
print("Best LR params:", random_lr.best_params_)
print("best score:", random_lr.best_score_)

#### Hyperparameter tuning was performed on the Logistic Regression model to identify the best regularization strength C. RandomizedSearchCV with 2 iterations and 2 fold cross validation was used. The value C=50 provided the best score for this dataset.

## 10. Model Comparison

- #### XGBoost achieved the highest f1 score, followed by Logistic Regression and SGD with slightly lower performance but faster training.
- #### All models perform extremely well on the majority class 0.
- #### XGBoost performs best on class 1 and 2, while SGD struggles a bit with class 1
- #### Among these models, Logistic Regression performs best in detecting the minority class 3, while XGBoost struggles significantly, often misclassifying class 3 as class 2.

## 11. Submission

In [ ]:
pipe_xgb.fit(X_train,y_train)
test_preds = pipe_xgb.predict(test_df)

In [ ]:
submission = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')
submission['label'] = test_preds
submission.to_csv('submission.csv', index=False)